[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/module-3-cleaning-and-exploring/notebook-3.2-fixing-data-types.ipynb)


# Module 3 — Fixing Data Types (Colab walkthrough)

A hands-on companion to the lecture on **fixing data types**, run on the same small, **synthetic** Barcelona-Bicing-style table of per-station snapshots.

## What this notebook is for

This notebook is a **preview** of what the lecture will show you. Run it top to bottom — or just read the printed outputs — to get a feel for the work before the lecture explains it: spotting when a column's *type* quietly hands you a wrong answer, converting it to the right type, and confirming the answer is now correct. You don't need to follow every line of code yet. The goal is to **picture what we'll be doing**, so when the lecture walks through the *why*, you already have a concrete example in your head. Every step prints a result you can check.

### How to run in Google Colab
1. `Runtime ▸ Run all`. The notebook **reads the dataset straight from this course's public GitHub repo** — nothing to upload, nothing to generate.
2. Everything below reads that CSV and prints a checkable result for each thing the lecture shows.

> ⚠️ **The data is synthetic.** Rows are fabricated to be *plausible* and to match the columns, types, quirks, and numbers used in the Module 3 lecture scripts. It is **not** real Bicing data — real Bicing snapshots are ~370 MB (too big for Colab) and no public trip-level Bicing data exists, which is why this small synthetic stand-in is hosted in the course repo and read directly above.


In [1]:
import numpy as np
import pandas as pd

# This course's practice data lives in its public GitHub repo. We read it straight
# from there, so the notebook runs the same in Google Colab or anywhere online —
# nothing to upload, nothing to generate.
DATA_URL = ("https://raw.githubusercontent.com/pmarcelino/mobillity-courses/main/"
            "mobillity-univ/module-3-cleaning-and-exploring/data/"
            "barcelona-bicing-synthetic/")


## Fixing Data Types

*The hook:* you ask for the busiest bike-share hour, the code runs with no error, and
returns **01:00**. The cause: a key column is still text. We load normally, **audit**
the dtypes, **convert**, then **verify**.


In [2]:
snap = pd.read_csv(f"{DATA_URL}station-snapshots-sample.csv")   # normal load
print("dtypes:")
print(snap.dtypes.to_string())
print()
for c in ["streetNumber", "updateTime", "nearbyStations"]:
    print(f"  {c:15s} -> {str(snap[c].dtype):8s} e.g. {snap[c].dropna().iloc[0]!r}")


dtypes:
id                  int64
type                  str
latitude          float64
longitude         float64
streetName            str
streetNumber          str
altitude            int64
slots               int64
bikes               int64
nearbyStations        str
status                str
updateTime            str

  streetNumber    -> str      e.g. '161'
  updateTime      -> str      e.g. '01/03/19 08:25:41'
  nearbyStations  -> str      e.g. '35,19'


In [3]:
# The silent failure: updateTime is TEXT in 'DD/MM/YY HH:MM:SS' form.
# A naive 'take the first two characters as the hour' actually grabs the DAY.
naive_busiest = snap["updateTime"].str[:2].value_counts().idxmax()
print("NAIVE busiest 'hour' (wrong):", naive_busiest, "  <- this is the day-of-month, not the hour")


NAIVE busiest 'hour' (wrong): 01   <- this is the day-of-month, not the hour


In [4]:
# Convert numeric-looking text; invalid tokens become NaN (errors='coerce').
street_num = pd.to_numeric(snap["streetNumber"], errors="coerce")
became_nan = int(street_num.isna().sum())
print("streetNumber rows -> NaN after to_numeric(coerce):", became_nan)
print("  offending tokens:",
      sorted(snap.loc[street_num.isna(), "streetNumber"].str.strip().unique().tolist()))

# Convert date text with the explicit European format.
update_dt = pd.to_datetime(snap["updateTime"], format="%d/%m/%y %H:%M:%S", errors="coerce")
print("updateTime rows that failed to parse (NaT):", int(update_dt.isna().sum()))

# Now the busiest hour is correct.
correct_busiest = int(update_dt.dt.hour.value_counts().idxmax())
print("CORRECT busiest hour (after fixing the type):", correct_busiest)

# Normalize the list-like text column into real lists.
nearby_lists = snap["nearbyStations"].fillna("").apply(
    lambda s: [x for x in str(s).split(",") if x])
print("nearbyStations normalized, e.g.:", nearby_lists.iloc[0])

print("re-audit -> streetNumber:", street_num.dtype, "| updateTime:", update_dt.dtype)


streetNumber rows -> NaN after to_numeric(coerce): 37
  offending tokens: ['23-25', 'SN', 's/n']
updateTime rows that failed to parse (NaT): 0
CORRECT busiest hour (after fixing the type): 8
nearbyStations normalized, e.g.: ['35', '19']
re-audit -> streetNumber: float64 | updateTime: datetime64[us]


## Done — fixing data types

The data-types lecture has run on the synthetic snapshots: the dtype **audit**, the
silent `01:00` vs the **correct** busiest hour (8) after fixing the type, the 37 invalid
`streetNumber` tokens coerced to NaN, and the list-in-text column normalized. Re-run any
time — the notebook reads the same published data, so the numbers don't change. Then try
the matching exercise yourself.
